In [30]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [31]:
import sys
from pathlib import Path
import json
import pandas as pd
import torch   
import os
from pathlib import Path

EMB_DIR = Path("noaug_data")

PROJECT_ROOT: /home/frosty/Programs/Python/Latent_Space_Vis
GNNGMM_DIR on sys.path: True
BACKEND_DIR on sys.path: False
Modules imported OK
Embeddings dir exists: True


In [32]:
layer_files = [
 'emb_train_unique_cas.pt',
 'output0_train_unique_cas.pt',
 'output1_train_unique_cas.pt',
 'output2_train_unique_cas.pt',
 'output3_train_unique_cas.pt',
 'output4_train_unique_cas.pt',
 'mlp_input_train_unique_cas.pt',
 'mlp_l1_train_unique_cas.pt',
 'mlp_l2_train_unique_cas.pt',
]
layer_names = [x.split('.')[0] for x in layer_files if x.split('.')[1] == "pt"]
layer_files_dict = dict(zip(layer_names, layer_files))

In [46]:
embeddings = [torch.load(f"{EMB_DIR}/{x}").squeeze().detach().numpy() for x in layer_files]

In [33]:
import gudhi
from gudhi.dtm_rips_complex import DTMRipsComplex
from sklearn.preprocessing import StandardScaler
sts = []
for x in layer_files:
    d = StandardScaler().fit_transform(torch.load(f"{EMB_DIR}/{x}").squeeze().detach().numpy())
    sc = DTMRipsComplex(points=d, k=1,q=2)
    st = sc.create_simplex_tree(max_dimension=1)
    st.collapse_edges()
    st.prune_above_filtration(3)
    st.compute_persistence()
    sts.append(st)

In [34]:
from latentlib.tracking import TDATracker, get_tracks_representatives
tracker = TDATracker(sts, 1)
tracks = get_tracks_representatives(tracker)

In [35]:
tracks.sort(key=lambda x: -len(x['representative_vertices']))

In [36]:
mapping = {i:0 for i in range(152)}
for x in tracks:
    for y in x['representative_vertices']:
        mapping[y] = x['track_id']

In [62]:
showScatter(embeddings, layer_names, id_mapping=mapping, imgFn=draw_mol)

In [37]:
# Call load_embeddings_from_files
# If your load_embeddings_from_files function supports passing only a directory, you can remove the second argument.
embeddings = load_embeddings_from_files(EMB_DIR, layer_files_dict)

print("Loaded layers:", list(embeddings.keys()))
for k, v in embeddings.items():
    try:
        print(k, getattr(v, "shape", (len(v),)))
    except Exception:
        print(k, type(v))

tmp_output_dir = PROJECT_ROOT / "frontend" / "public" / "data" / "tmp_notebook"
tmp_output_dir.mkdir(parents=True, exist_ok=True)

pipeline = EmbeddingPipeline(
    embeddings=embeddings,
    output_dir=tmp_output_dir,
    #id_mapping=mapping,    
    n_components=4
)

print("Fitting GMMs...")
pipeline.fit_gmms()
print("Computing MDS / distance embedding...")
pipeline.compute_mds()

scatter_df = pipeline.generate_scatter_plot_data()
sankey_df = pipeline.generate_sankey_data()

print("scatter rows:", len(scatter_df))
print("sankey rows:", len(sankey_df))

display(scatter_df.head())
display(sankey_df.head())

  ✓ emb_train_unique_cas: shape (139, 128)
  ✓ output0_train_unique_cas: shape (139, 256)
  ✓ output1_train_unique_cas: shape (139, 256)
  ✓ output2_train_unique_cas: shape (139, 256)
  ✓ output3_train_unique_cas: shape (139, 256)
  ✓ output4_train_unique_cas: shape (139, 256)
  ✓ mlp_input_train_unique_cas: shape (139, 260)
  ✓ mlp_l1_train_unique_cas: shape (139, 130)
  ✓ mlp_l2_train_unique_cas: shape (139, 130)
Loaded layers: ['emb_train_unique_cas', 'output0_train_unique_cas', 'output1_train_unique_cas', 'output2_train_unique_cas', 'output3_train_unique_cas', 'output4_train_unique_cas', 'mlp_input_train_unique_cas', 'mlp_l1_train_unique_cas', 'mlp_l2_train_unique_cas']
emb_train_unique_cas (139, 128)
output0_train_unique_cas (139, 256)
output1_train_unique_cas (139, 256)
output2_train_unique_cas (139, 256)
output3_train_unique_cas (139, 256)
output4_train_unique_cas (139, 256)
mlp_input_train_unique_cas (139, 260)
mlp_l1_train_unique_cas (139, 130)
mlp_l2_train_unique_cas (139, 13

,id,x,y,color_value,layer
0,0,1031.689886,1194.505807,0.0,emb_train_unique_cas
1,1,1517.206040,847.448528,0.0,emb_train_unique_cas
2,2,572.806456,-2706.470331,0.0,emb_train_unique_cas
3,3,3038.048710,642.871445,0.0,emb_train_unique_cas
4,4,1321.391055,-323.415728,0.0,emb_train_unique_cas


,real_id,from_layer,to_layer,from_cluster,to_cluster
0,0,emb_train_unique_cas,mlp_input_train_unique_cas,2,1
1,1,emb_train_unique_cas,mlp_input_train_unique_cas,3,1
2,2,emb_train_unique_cas,mlp_input_train_unique_cas,0,1
3,3,emb_train_unique_cas,mlp_input_train_unique_cas,1,1
4,4,emb_train_unique_cas,mlp_input_train_unique_cas,2,1


In [38]:
import pandas as pd
smiles = pd.read_csv(f"{EMB_DIR}/train_unique_cas.csv")

In [39]:
from rdkit import Chem
from rdkit.Chem import Draw
import matplotlib.pyplot as plt
import base64
import io

def draw_mol(idx):
    mol = Chem.MolFromSmiles(smiles.iloc[idx]['ligand_1_smiles'])
    img = Draw.MolToImage(mol)
    
    buffered = io.BytesIO()
    img.save(buffered, format="PNG")
    img_bytes = buffered.getvalue()
    img_base64_bytes = base64.b64encode(img_bytes)
    return img_base64_bytes.decode('ascii')

In [40]:
from latentvis import showSankey, showScatter

In [41]:
showSankey(sankey_df.to_dict(orient="records"))

In [ ]:
showScatter(scatter_df.to_dict(orient="records"), draw_mol)